# CV Screening Multi-Agent Demo

This notebook is a Colab/Kaggle launcher for the existing project code. It does not modify the source files. Run the cells from top to bottom to install dependencies, verify the project files, and execute the deterministic CV screening pipeline.

## 1. Runtime Check

Colab and Kaggle usually start inside the repository folder after upload. If this notebook is not in the project root, set `PROJECT_ROOT` to the folder that contains `src`, `data`, `models`, and `requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

def looks_like_project_root(path: Path) -> bool:
    return (path / "src" / "main.py").exists() and (path / "requirements.txt").exists()


PROJECT_ROOT = Path.cwd()
if not looks_like_project_root(PROJECT_ROOT):
    candidates = [path for path in Path.cwd().rglob("requirements.txt") if looks_like_project_root(path.parent)]
    candidates += [path for path in Path("/content").glob("**/requirements.txt") if looks_like_project_root(path.parent)] if Path("/content").exists() else []
    candidates += [path for path in Path("/kaggle/input").glob("**/requirements.txt") if looks_like_project_root(path.parent)] if Path("/kaggle/input").exists() else []
    if candidates:
        PROJECT_ROOT = candidates[0].parent

os.chdir(PROJECT_ROOT)

required_paths = [
    PROJECT_ROOT / "src" / "main.py",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "data" / "sample_cvs.json",
    PROJECT_ROOT / "data" / "sample_jobs.json",
]

missing = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Notebook is not running from the project root, or files are missing: " + ", ".join(missing)
    )

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Files OK")

## 2. Install Dependencies

The deterministic run does not require Ollama. The project includes CrewAI dependencies, but this notebook runs the deterministic backend because it is the most reliable option in hosted notebooks.

In [ ]:
%pip install -q -r requirements.txt

## 3. Train Or Verify The Model

If `models/cv_fit_model.pt` is already included, this cell keeps it. If it is missing, it trains the baseline model using the existing project code.

In [ ]:
from pathlib import Path

model_path = Path("models/cv_fit_model.pt")
if model_path.exists():
    print(f"Model found: {model_path}")
else:
    print("Model not found. Training baseline model...")
    !python -m src.train_baseline

## 4. Run Single CV Screening

This uses the existing sample CV and job description from the `data` folder.

In [ ]:
!python -m src.main --runtime deterministic

## 5. Run Batch Screening

This screens every CV in `data/sample_cvs.json` against the selected sample job.

In [ ]:
!python -m src.main --runtime deterministic --batch-screening --cv-file data/sample_cvs.json --job-file data/sample_jobs.json

## 6. Optional Evaluation

Run this cell if you want to regenerate the pipeline evaluation output.

In [ ]:
!python -m src.evaluate_pipeline

## 7. View Generated Logs

The pipeline writes JSONL audit logs into the `logs` folder.

In [ ]:
from pathlib import Path

log_files = sorted(Path("logs").glob("*.jsonl"))
print(f"Found {len(log_files)} log file(s).")
for path in log_files[-5:]:
    print(path)